In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

CHOICES = ["A", "B", "C", "D"]
QUERY_TEMPLATE_MULTICHOICE = """
Answer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.

{Question}

A) {A}
B) {B}
C) {C}
D) {D}
""".strip()

ANSWER_PATTERN_MULTICHOICE = r"(?i)Answer[ \t]*:[ \t]*\$?([A-D])\$?"
# source: https://github.com/openai/simple-evals/blob/main/common.py

In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048, # Choose any for long context!
    # load_in_4bit = False,  # 4 bit quantization to reduce memory
    # load_in_8bit = True, # [NEW!] A bit more accurate, uses 2x memory
    # full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

/data/common/ethanchang/ucct/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.54.0.dev0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.438 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [3]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [4]:
from datasets import load_dataset
## TEST             : 14K   ROWS
## AUXILIARY TRAIN  : 99.8K ROWS
## VALIDATION       : 1.53K ROWS
dataset = load_dataset("cais/mmlu", "all", split="validation")

In [5]:
# from unsloth.chat_templates import standardize_data_formats
# dataset = standardize_data_formats(dataset)

In [6]:
dataset[0]

{'question': 'The cyclic subgroup of Z_24 generated by 18 has order',
 'subject': 'abstract_algebra',
 'choices': ['4', '8', '12', '6'],
 'answer': 0}

In [11]:
# apply chat templates
def formatting_prompts_func(examples):
    query = QUERY_TEMPLATE_MULTICHOICE.format(Question=examples["question"], **{choice: text for choice, text in zip(CHOICES, examples["choices"])})  # examples["question"]
    text = tokenizer.apply_chat_template([{"role" : "user", "content": query}], tokenize = False, add_generation_prompt = True)
    return { "text" :  text }

dataset = dataset.map(formatting_prompts_func, batched = False)
dataset[100]

Map: 100%|██████████| 1531/1531 [00:00<00:00, 5398.34 examples/s]


{'question': 'Reduction of D-xylose with NaBH4 yields a product that is a',
 'subject': 'college_chemistry',
 'choices': ['racemic mixture',
  'single pure enantiomer',
  'mixture of two diastereomers in equal amounts',
  'meso compound'],
 'answer': 3,
 'text': "<|im_start|>user\nAnswer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.\n\nReduction of D-xylose with NaBH4 yields a product that is a\n\nA) racemic mixture\nB) single pure enantiomer\nC) mixture of two diastereomers in equal amounts\nD) meso compound<|im_end|>\n<|im_start|>assistant\n"}

In [76]:
from transformers import TextStreamer  # type: ignore

streamer = TextStreamer(tokenizer, skip_prompt=False)
text = dataset[100]["text"]  # type: ignore
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=1000,  # Increase for longer outputs!
    temperature=0.7,
    top_p=0.8,
    top_k=20,  # For non thinking
    streamer=streamer,
)

<|im_start|>user
Answer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.

Reduction of D-xylose with NaBH4 yields a product that is a

A) racemic mixture
B) single pure enantiomer
C) mixture of two diastereomers in equal amounts
D) meso compound<|im_end|>
<|im_start|>assistant
To determine the product of the reduction of D-xylose with NaBH4, we need to understand the nature of the reaction and the starting material.

D-xylose is an aldopentose with multiple chiral centers. When an aldehyde group (–CHO) is reduced using NaBH4, it becomes a primary alcohol (–CH2OH). This reduction occurs at the carbonyl carbon, which was originally the aldehyde carbon in D-xylose.

In D-xylose, the aldehyde group is at C1, and the molecule has chiral centers at C2, C3, and C4. After reduction, the aldehyde at C1 becomes a –CH2OH group. The chiral centers at C2, C3, and C4 remain unchan

In [78]:
from itertools import accumulate, islice

examples = [
    """<|im_start|>user
Answer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.

Reduction of D-xylose with NaBH4 yields a product that is a

A) racemic mixture
B) single pure enantiomer
C) mixture of two diastereomers in equal amounts
D) meso compound<|im_end|>
<|im_start|>assistant
Answer: B<|im_end|>
    """.strip(),
    """
<|im_start|>user
Answer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.

The Rydberg equation v = R_H(1/n_1^2 - 1/n_2^2) accurately predicts the UV-visible emission spectrum of the hydrogen atom. A form of the Rydberg equation may also be used to predict the UV-visible emission for all of the following EXCEPT

A) hydride ion, H−
B) deuterium atom, D
C) tritium atom, T
D) helium cation, He+<|im_end|>
<|im_start|>assistant
Answer: A<|im_end|>
    """.strip(),
    """<|im_start|>user
Answer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.

The 13C spectrum of which isomer of C6H14 has lines with three distinct chemical shifts?

A) hexane
B) 2-methylpentane
C) 3-methylpentane
D) 2,3-dimethylbutane<|im_end|>
<|im_start|>assistant
Answer: C<|im_end|>
    """.strip(),
]


def getRanges(lengths):
    def addAndHoldPrev(state, ele):
        _, total = state
        return total, total + ele

    return islice(
        accumulate(lengths, func=addAndHoldPrev, initial=(0, 0)), 1, None
    )  # remove initial (0, 0)


lengths = [len(tokenizer(ex)["input_ids"]) for ex in examples]
examplesAndRanges = [entry for entry in zip(examples, getRanges(lengths))]

text = "".join(examples) + dataset[0]["text"] # type: ignore
out = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1000, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = streamer
)
tokenizer.batch_decode(out)



<|im_start|>user
Answer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.

Reduction of D-xylose with NaBH4 yields a product that is a

A) racemic mixture
B) single pure enantiomer
C) mixture of two diastereomers in equal amounts
D) meso compound<|im_end|>
<|im_start|>assistant
Answer: B<|im_end|><|im_start|>user
Answer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.

The Rydberg equation v = R_H(1/n_1^2 - 1/n_2^2) accurately predicts the UV-visible emission spectrum of the hydrogen atom. A form of the Rydberg equation may also be used to predict the UV-visible emission for all of the following EXCEPT

A) hydride ion, H−
B) deuterium atom, D
C) tritium atom, T
D) helium cation, He+<|im_end|>
<|im_start|>assistant
Answer: A<|im_end|><|im_start|>user

The order of a cyclic subgroup in Z_n generated by an element k is given by n / gcd(n, k).

Here, n = 24 and k = 18.  
gcd(24, 18) = 6.  
So the order of the subgroup is 24 / 6 = 4.

Answer: A<|im_end|>


["<|im_start|>user\nAnswer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.\n\nReduction of D-xylose with NaBH4 yields a product that is a\n\nA) racemic mixture\nB) single pure enantiomer\nC) mixture of two diastereomers in equal amounts\nD) meso compound<|im_end|>\n<|im_start|>assistant\nAnswer: B<|im_end|><|im_start|>user\nAnswer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) where LETTER is one of ABCD.\n\nThe Rydberg equation v = R_H(1/n_1^2 - 1/n_2^2) accurately predicts the UV-visible emission spectrum of the hydrogen atom. A form of the Rydberg equation may also be used to predict the UV-visible emission for all of the following EXCEPT\n\nA) hydride ion, H−\nB) deuterium atom, D\nC) tritium atom, T\nD) helium cation, He+<|im_end|>\n<|im_start|>assistant\nAnswer: A<|im

In [ ]:
def attachHooks(model, layers, all_layer_outputs):
    num_layers = len(layers)
    
    def get_layer_output_hook(layer_idx):
        def hook(module, input, output):
            hidden_states = output[0]
            if hidden_states.shape[1] > 1:
                all_layer_outputs[layer_idx] = hidden_states.detach()
        return hook

    hook_handles = []
    for i in range(num_layers):
        target_layer = layers[i]
        handle = target_layer.register_forward_hook(get_layer_output_hook(i))
        hook_handles.append(handle)
    print(f"Attached {len(hook_handles)} hooks to layers 0 through {num_layers - 1}.")

In [ ]:
# len([l for l in model.named_modules()])

547

In [43]:
import torch
import torch.nn as nn

def findModuleList(model):
    # 1. Try to find the container that holds the blocks
    
    # Heuristic: The main container is usually a ModuleList with many children
    largest_module_list = None
    max_len = 0

    for name, module in model.named_modules():
        if isinstance(module, nn.ModuleList):
            # We assume the main "spine" is the longest list in the model
            if len(module) > max_len:
                max_len = len(module)
                largest_module_list = module
                
    if largest_module_list is None:
        print("Warning: Could not automatically find the block container.")
        return

    # 2. Register hooks on the children of that list
    print(f"Found main block container with {len(largest_module_list)} layers.")
    
    return largest_module_list

# for layer in findModuleList(model):
#     print(la)

transformer_decoder_layers = findModuleList(model)

# for name, module in findModuleList(model).named_children():
#     print(module)
#     print("\n\n")

Found main block container with 36 layers.


In [44]:
for name, module in transformer_decoder_layers[0].named_children():
    print(name)

self_attn
mlp
input_layernorm
post_attention_layernorm


In [45]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0): Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear4bit(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Q